# OhioT1DM Test Pipeline

This notebook processes the **test** split only.

It extracts CGM values from OhioT1DM XML files, saves CSV outputs, generates figures, and packages a GitHub-ready ZIP.

Output folder: `test_outputs/`


In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Configure paths

In [ ]:
# Change this only if your extracted OhioT1DM folder is elsewhere.
SPLIT_NAME = "test"
OHIO_SEARCH_ROOT = "/content"
OUTDIR = f"/content/{SPLIT_NAME}_outputs"

os.makedirs(OUTDIR, exist_ok=True)
print("Output folder:", OUTDIR)


## 2. Find XML folders automatically

In [ ]:
def find_split_folders(search_root, split_name):
    """
    Finds folders like:
    /content/.../2018/train
    /content/.../2020/train
    or test equivalents.
    """
    matched = []
    for root, dirs, files in os.walk(search_root):
        normalized = root.replace("\\", "/").lower()
        if normalized.endswith(f"/2018/{split_name}") or normalized.endswith(f"/2020/{split_name}"):
            if any(f.lower().endswith(".xml") for f in files):
                matched.append(root)
    return sorted(matched)

folders = find_split_folders(OHIO_SEARCH_ROOT, SPLIT_NAME)

print("Detected folders:")
for f in folders:
    print("-", f)

if not folders:
    xml_files = glob.glob("/content/**/*.xml", recursive=True)
    print("No split folders detected.")
    print("XML files found directly:", len(xml_files))
    print(xml_files[:10])


## 3. Extract CGM rows from XML

In [ ]:
def extract_cgm_from_xml(xml_file, split_name):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    rows = []

    for glucose_block in root.iter("glucose_level"):
        for event in glucose_block.iter("event"):
            ts = event.attrib.get("ts")
            value = event.attrib.get("value")

            if ts is not None and value is not None:
                rows.append({
                    "timestamp": ts,
                    "glucose": float(value),
                    "source_file": os.path.basename(xml_file),
                    "split": split_name,
                    "xml_path": xml_file,
                })

    return pd.DataFrame(rows)


def load_split(folders, split_name):
    frames = []

    for folder in folders:
        xml_files = sorted(glob.glob(os.path.join(folder, "*.xml")))
        print(f"Folder: {folder} | XML files: {len(xml_files)}")

        for xml_file in xml_files:
            df = extract_cgm_from_xml(xml_file, split_name)

            if not df.empty:
                df["dataset_folder"] = folder
                frames.append(df)
                print("Loaded:", os.path.basename(xml_file), df.shape)
            else:
                print("No CGM rows:", os.path.basename(xml_file))

    if frames:
        return pd.concat(frames, ignore_index=True)

    return pd.DataFrame(columns=[
        "timestamp", "glucose", "source_file", "split", "xml_path", "dataset_folder"
    ])


## 4. Load Test data

In [ ]:
df = load_split(folders, SPLIT_NAME)

if df.empty:
    raise ValueError("No test CGM data loaded. Check folder upload/path.")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp", "glucose"])
df = df.sort_values(["source_file", "timestamp"]).reset_index(drop=True)

print("Rows:", len(df))
print("Files:", df["source_file"].nunique())
df.head()


## 5. Save CSV outputs

In [ ]:
cgm_path = os.path.join(OUTDIR, "ohiot1dm_test_cgm.csv")
df.to_csv(cgm_path, index=False)

summary_by_file = (
    df.groupby("source_file")
      .agg(
          records=("glucose", "count"),
          mean_glucose=("glucose", "mean"),
          std_glucose=("glucose", "std"),
          min_glucose=("glucose", "min"),
          max_glucose=("glucose", "max"),
          start_time=("timestamp", "min"),
          end_time=("timestamp", "max"),
      )
      .reset_index()
)

summary_by_file["mean_glucose"] = summary_by_file["mean_glucose"].round(2)
summary_by_file["std_glucose"] = summary_by_file["std_glucose"].round(2)

summary_file_path = os.path.join(OUTDIR, "ohiot1dm_test_summary_by_file.csv")
summary_by_file.to_csv(summary_file_path, index=False)

overall_summary = pd.DataFrame([{
    "split": SPLIT_NAME,
    "total_records": len(df),
    "number_of_files": df["source_file"].nunique(),
    "mean_glucose": round(df["glucose"].mean(), 2),
    "std_glucose": round(df["glucose"].std(), 2),
    "min_glucose": round(df["glucose"].min(), 2),
    "max_glucose": round(df["glucose"].max(), 2),
}])

overall_path = os.path.join(OUTDIR, "ohiot1dm_test_overall_summary.csv")
overall_summary.to_csv(overall_path, index=False)

print(cgm_path)
print(summary_file_path)
print(overall_path)
overall_summary


## 6. Generate figures

In [ ]:
sample_file = df["source_file"].iloc[0]
sample = df[df["source_file"] == sample_file].copy().sort_values("timestamp")

start = sample["timestamp"].min()
end = start + pd.Timedelta(hours=24)
sample_24h = sample[(sample["timestamp"] >= start) & (sample["timestamp"] <= end)].copy()

plt.figure(figsize=(8, 4))
plt.plot(sample_24h["timestamp"], sample_24h["glucose"], linewidth=1.5)
plt.xlabel("Time")
plt.ylabel("Glucose (mg/dL)")
plt.title("OhioT1DM Test CGM Sample - 24 Hours")
plt.xticks(rotation=30)
plt.tight_layout()
plot_24h_path = os.path.join(OUTDIR, "ohiot1dm_test_24h_plot.png")
plt.savefig(plot_24h_path, dpi=300)
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(df["glucose"], bins=40)
plt.xlabel("Glucose (mg/dL)")
plt.ylabel("Frequency")
plt.title("OhioT1DM Test Glucose Distribution")
plt.tight_layout()
dist_path = os.path.join(OUTDIR, "ohiot1dm_test_glucose_distribution.png")
plt.savefig(dist_path, dpi=300)
plt.show()

print(plot_24h_path)
print(dist_path)


## 7. Counterfactual-style demonstration from real CGM sample

In [ ]:
sample_24h["hours"] = (
    sample_24h["timestamp"] - sample_24h["timestamp"].min()
).dt.total_seconds() / 3600

real_baseline = sample_24h["glucose"].values
reduced_carbs_est = real_baseline - 0.15 * np.maximum(real_baseline - 110, 0)
walking_est = real_baseline - 0.10 * np.maximum(real_baseline - 110, 0)

cf_df = pd.DataFrame({
    "hours": sample_24h["hours"].values,
    "observed_cgm": real_baseline,
    "estimated_reduced_carbs": reduced_carbs_est,
    "estimated_walking_effect": walking_est,
})

cf_csv_path = os.path.join(OUTDIR, "ohiot1dm_test_counterfactual_demo.csv")
cf_df.to_csv(cf_csv_path, index=False)

plt.figure(figsize=(7, 4))
plt.plot(cf_df["hours"], cf_df["observed_cgm"], label="Observed CGM", linewidth=2)
plt.plot(cf_df["hours"], cf_df["estimated_reduced_carbs"], label="Estimated reduced carbs", linewidth=2)
plt.plot(cf_df["hours"], cf_df["estimated_walking_effect"], label="Estimated walking effect", linewidth=2)
plt.xlabel("Time (hours)")
plt.ylabel("Glucose (mg/dL)")
plt.title("OhioT1DM Test Real CGM-Based Counterfactual Demo")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
cf_plot_path = os.path.join(OUTDIR, "ohiot1dm_test_counterfactual_demo.png")
plt.savefig(cf_plot_path, dpi=300)
plt.show()

print(cf_csv_path)
print(cf_plot_path)


## 8. Package for GitHub upload

In [ ]:
github_dir = f"/content/github_test_upload"
data_dir = os.path.join(github_dir, "data", SPLIT_NAME)
fig_dir = os.path.join(github_dir, "figures", SPLIT_NAME)

os.makedirs(data_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

for file in glob.glob(os.path.join(OUTDIR, "*.csv")):
    shutil.copy(file, data_dir)

for file in glob.glob(os.path.join(OUTDIR, "*.png")):
    shutil.copy(file, fig_dir)

readme = """# OhioT1DM Test Outputs

This folder contains test split outputs generated from the OhioT1DM XML processing notebook.

## Included
- Extracted CGM CSV
- File-level summary CSV
- Overall summary CSV
- 24-hour CGM plot
- Glucose distribution plot
- Counterfactual-style demonstration output

## Note
Raw OhioT1DM XML files are not included here. Use of derived CSV files should follow the original OhioT1DM dataset license terms.
"""

with open(os.path.join(data_dir, "README.md"), "w") as f:
    f.write(readme)

zip_path = shutil.make_archive(f"/content/github_test_upload", "zip", github_dir)
print("Created:", zip_path)


In [ ]:
from google.colab import files
files.download(zip_path)
